# Cave Exploration RL Training

This notebook contains the cave exploration reinforcement learning training pipeline, organized into separate cells for better modularity and experimentation.

# Imports etc.

In [3]:

import os
import sys
# Add execution tracking to debug duplicate output
print("=== STARTING IMPORTS CELL ===")

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# Configure JAX GPU memory settings BEFORE importing jax - OPTIMIZED FOR 40GB A100
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.99'  # Use 98% of GPU memory (~39.2GB out of 40GB)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'  # Use platform allocator
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'  # Don't preallocate - grow as needed to avoid fragmentation
#os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Allow dynamic growth

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

import jax
from jax import numpy as jp
from jax.lib import xla_bridge

print("Device count: ", jax.device_count())

# Configure JAX to use only GPU1

print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"JAX memory fraction set to: {os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION')}")
print(f"JAX preallocate disabled: {os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE')}")

print("JAX backend info:")
print(f"Platform: {xla_bridge.get_backend().platform}")
print(f"Device count: {xla_bridge.get_backend().device_count()}")
print(f"Devices: {xla_bridge.get_backend().devices()}")

# JAX configuration optimized for large workloads
jax.config.update('jax_enable_x64', False)  # Use float32 to save memory
jax.config.update('jax_traceback_filtering', 'off')
# Use bfloat16 for even better memory efficiency (optional - comment out if you need float32 precision)
# jax.config.update('jax_default_matmul_precision', 'bfloat16')

# Check GPU availability and memory
gpu_available = jax.devices()[0].platform == 'gpu'
print(f"GPU available: {gpu_available}")

if gpu_available:
    gpu_device = jax.devices('gpu')[0]
    print(f"GPU device: {gpu_device}")
else:
    print("No GPU device found.")

import signal
import json
import functools
import mujoco
from datetime import datetime
from pathlib import Path
import imageio
import gc

print("Basic imports completed...")

# Brax and training imports
from brax.io import model
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from flax.training import orbax_utils
from orbax import checkpoint as ocp
from mujoco_playground.config import locomotion_params
from mujoco_playground import wrapper
from tensorboardX import SummaryWriter

print("Brax imports completed...")

# Task-specific imports
from tasks.cave_exploration.cave_exploration import CaveExplore
from tasks.common.randomize import domain_randomize as reachbot_randomize
from utils.telegram_messenger import send_message_sync

print("Task-specific imports completed...")

# Global variables
ENV_STR = 'Go1JoystickFlatTerrain'
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

# Signal handler for graceful interruption
def signal_handler(sig, frame):
    print('Program exited via keyboard interrupt')
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)

# JSON encoder for JAX arrays
class JaxArrayEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, jp.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

print("=== ALL IMPORTS LOADED SUCCESSFULLY! ===")

=== STARTING IMPORTS CELL ===


ModuleNotFoundError: No module named 'jax'

In [ ]:
# SSH Disconnection Resilience - Keep Running Independent of SSH Session
import signal
import os

print("=== SETTING UP SSH DISCONNECTION RESILIENCE ===")

def ignore_sighup(signum, frame):
    """Ignore SIGHUP signal (SSH disconnection) to keep process running"""
    print(f"\n🔌 SSH disconnection detected (SIGHUP), but continuing to run in background...")
    print(f"📋 Process PID: {os.getpid()}")
    print("🚀 Training will continue independently!")

def handle_sigterm(signum, frame):
    """Handle SIGTERM gracefully but continue for most cases"""
    print(f"\n⚠️  Received SIGTERM, but attempting to continue...")
    print("🔄 If this is a system shutdown, the process will be force-killed anyway")

def handle_sigint(signum, frame):
    """Handle Ctrl+C - this one we do want to respect for manual interruption"""
    print(f"\n🛑 Received SIGINT (Ctrl+C) - User requested interruption")
    print("💾 Cleaning up gracefully...")
    exit(0)

# Set up signal handlers
signal.signal(signal.SIGHUP, ignore_sighup)    # SSH disconnection - IGNORE
signal.signal(signal.SIGTERM, handle_sigterm)  # Termination - TRY TO IGNORE  
signal.signal(signal.SIGINT, handle_sigint)    # Ctrl+C - RESPECT

# Additional resilience measures
print("🛡️  SSH Disconnection Resilience Active!")
print(f"📋 Process PID: {os.getpid()}")
print("🔌 SSH disconnections will be ignored")
print("🚀 Notebook will continue running in background")
print("⚠️  Note: Only Ctrl+C will stop the training")

# Optional: Redirect stdout/stderr to files for monitoring after SSH disconnect
import sys
from datetime import datetime

# Create a log file for output monitoring
log_file = f"notebook_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print(f"📝 Consider tailing this log file after SSH reconnection: {log_file}")

class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_path = os.path.join(os.getcwd(), log_file)
log_f = open(log_path, 'a')
sys.stdout = Tee(sys.__stdout__, log_f)
sys.stderr = Tee(sys.__stderr__, log_f)
print(f"All notebook output will be written to both the notebook and: {log_path}")

print("✅ SSH resilience setup complete!")

=== SETTING UP SSH DISCONNECTION RESILIENCE ===
🛡️  SSH Disconnection Resilience Active!
📋 Process PID: 2872456
🔌 SSH disconnections will be ignored
🚀 Notebook will continue running in background
⚠️  Note: Only Ctrl+C will stop the training
📝 Consider tailing this log file after SSH reconnection: notebook_output_20250722_000605.log
All notebook output will be written to both the notebook and: /home/ga53voq/master_thesis/tasks/cave_exploration/notebook_output_20250722_000605.log
✅ SSH resilience setup complete!


## 🔌 SSH Disconnection Recovery Guide

If your SSH connection gets interrupted, here's how to check if your training is still running and monitor progress:

### 1. Check if the process is still running:
```bash
ps aux | grep python | grep cave_exploration
# or
ps aux | grep jupyter
```

### 2. Monitor the training logs:
```bash
# Check TensorBoard logs
ls -la logs/cave_exploration-*/
tail -f logs/cave_exploration-*/events.out.tfevents.*

# Monitor any output files
tail -f notebook_output_*.log
```

### 3. Check GPU usage (confirms training is active):
```bash
nvidia-smi
# or watch it continuously
watch -n 1 nvidia-smi
```

### 4. Kill the process if needed:
```bash
# Find the PID
ps aux | grep python | grep cave_exploration
# Kill gracefully
kill -TERM <PID>
# Force kill if needed
kill -9 <PID>
```

### 5. Alternative: Use tmux/screen for future sessions:
```bash
# Start a tmux session before running notebook
tmux new-session -d -s training
tmux attach -t training
# Then run your notebook - it will survive SSH disconnections
```

**The notebook is now configured to ignore SSH disconnections and continue training!** 🚀

# Environment configuration

In [ ]:
# Environment configuration
from tasks.cave_exploration.cave_exploration import default_config as reachbot_config

env_cfg = reachbot_config()

# Basic simulation parameters
env_cfg.sim_dt = 0.004
env_cfg.action_scale = 1

# PID control parameters
env_cfg.Kp_pri = 60.0
env_cfg.Kd_pri = 20.0
env_cfg.Kp_rot = 25.0
env_cfg.Kd_rot = 2.0

env_cfg.noise_config.level = 0.0

# Reward scaling configuration
env_cfg.reward_config.scales.orientation = -0.5
env_cfg.reward_config.scales.torques = 0.0#-0.00005
env_cfg.reward_config.scales.action_rate = 0.0#-0.0001
env_cfg.reward_config.scales.dof_pos_limits = 0.0#-0.05
env_cfg.reward_config.scales.energy = 0.0#-0.00001
env_cfg.reward_config.scales.termination = -1.0
env_cfg.reward_config.scales.inactivity = -0.1#0.1

# Target-based rewards
env_cfg.reward_config.scales.distance_from_start = -0.1#5.0
env_cfg.reward_config.scales.track_lidar_direction = 1.0
env_cfg.reward_config.scales.stability = 0.1
env_cfg.reward_config.scales.exploration_rate = 0.0

print("Environment configuration completed!")

# PPO parameters

In [5]:
# PPO training parameters configuration
ppo_params = locomotion_params.brax_ppo_config(ENV_STR)
ppo_training_params = dict(ppo_params)

# Modify params for training
ppo_training_params["num_timesteps"] = 50_000_000
ppo_training_params["episode_length"] = 5000
ppo_training_params["num_envs"] = 4096#2048 
ppo_training_params["batch_size"] = 256#1024
ppo_training_params["num_minibatches"] = 32
ppo_training_params["num_updates_per_batch"] = 4
ppo_training_params["unroll_length"] = 64
ppo_training_params["entropy_cost"] = 1e-2
ppo_training_params["learning_rate"] = 3e-4
ppo_training_params["discounting"] = 0.995
#ppo_training_params["clipping_epsilon"] = 0.1
ppo_training_params["num_evals"] = ppo_training_params["num_timesteps"] // 10_000_000
if (ppo_training_params["num_evals"] < 10):
    ppo_training_params["num_evals"] = 10


print("PPO training parameters:")
for key, value in ppo_training_params.items():
    print(f"  {key}: {value}")

print("\nPPO parameters configuration completed!")

# Training 


In [ ]:
import os
import threading
print(f"🔥 PID: {os.getpid()} | Thread: {threading.current_thread().ident} | Time: {datetime.now()}")

# Training execution
# Create log directory for training run
datetime_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
logdir = os.path.join(os.getcwd(), "logs/cave_exploration-"+datetime_str)
os.makedirs(logdir, exist_ok=True)

# Create environment
env = CaveExplore(config=env_cfg)

# Initialize tracking variables
timesteps = []
rewards = []
total_rewards = []
total_rewards_std = []
times = [datetime.now()]

writer = SummaryWriter(logdir=logdir)

caveIds = env.caveIds
print(caveIds)

# Save configurations
print("Saving configs")

# Convert ConfigDict to regular dict recursively
def convert_to_dict(obj):
    """Convert ConfigDict and other non-serializable objects to regular dicts"""
    if hasattr(obj, 'to_dict'):
        # Handle ConfigDict objects
        return convert_to_dict(obj.to_dict())
    elif hasattr(obj, '__dict__'):
        # Handle objects with __dict__ attribute
        return convert_to_dict(obj.__dict__)
    elif isinstance(obj, dict):
        return {k: convert_to_dict(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_dict(v) for v in obj]
    elif isinstance(obj, jp.ndarray):
        return obj.tolist()
    elif isinstance(obj, float) and obj == float('inf'):
        return 1e308
    elif isinstance(obj, float) and obj == float('-inf'):
        return -1e308
    else:
        return obj

configs = {
    "env_cfg": convert_to_dict(env_cfg),
    "ppo_params": convert_to_dict(ppo_training_params),
    "cave_ids": convert_to_dict(caveIds),
}

config_path = os.path.join(logdir, 'config.json')
with open(config_path, "w", encoding="utf-8") as fp:
    json.dump(configs, fp, indent=4, cls=JaxArrayEncoder)
print(f"Configuration saved to {config_path}")
writer.add_text('config', json.dumps(configs, indent=4))

# Progress tracking function
def progress(num_steps, metrics):
    """Function to track progress and log metrics during training."""
    print(f"Progress at step {num_steps}: {metrics}")
    # Log to TensorBoard
    for key, value in metrics.items():
        if not (jp.isnan(value) or jp.isinf(value)):
            writer.add_scalar(key, value, num_steps)
        else:
            print(f"Warning: Skipping NaN/Inf value for metric '{key}' at step {num_steps}")

    if "eval/episode_reward" in metrics:
        episode_reward = metrics["eval/episode_reward"]
        if jp.isnan(episode_reward) or jp.isinf(episode_reward):
            print("Warning: NaN/Inf reward encountered, aborting.")
            run_duration = str(datetime.now() - times[0])
            send_message_sync(
                task="Cave Exploration RL Training",
                duration=run_duration,
                result="Failed: NaN/Inf reward encountered"
            )
            raise ValueError(f"NaN/Inf reward encountered at step {num_steps}: {episode_reward}")
        
        times.append(datetime.now())
        timesteps.append(num_steps)
        total_rewards.append(episode_reward)
        total_rewards_std.append(metrics["eval/episode_reward_std"])
    
        writer.flush()
        metrics["timesteps"] = num_steps
        metrics["time"] = (times[-1] - times[0]).total_seconds()
        rewards.append(metrics)
        
        percent_complete = (num_steps / ppo_training_params["num_timesteps"]) * 100
        if num_steps == 0:
            remaining_time_str = "unknown"
        else:
            remaining_time = (ppo_training_params["num_timesteps"] - num_steps) * (times[-1] - times[0]).total_seconds() / num_steps / 60
            remaining_time_str = f"{remaining_time:.2f}"
        
        print(f"step: {num_steps}/{ppo_training_params['num_timesteps']} ({percent_complete:.1f}%), reward: {total_rewards[-1]:.3f} +/- {total_rewards_std[-1]:.3f}, time passed (min): {(times[-1] - times[0]).total_seconds() / 60:.2f} min, calculated time left (min): {remaining_time_str} min")

# Network factory setup
print("Input layer size:", env.observation_size)
print("Output layer size:", env.action_size)
network_factory = ppo_networks.make_ppo_networks(observation_size=env.observation_size, action_size=env.action_size)
if "network_factory" in ppo_params:
    if "network_factory" in ppo_training_params:
        del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory
    )

# Checkpoint saving function
def policy_params_fn(current_step, make_policy, params):
    del make_policy  # Unused.
    orbax_checkpointer = ocp.PyTreeCheckpointer()
    save_args = orbax_utils.save_args_from_target(params)
    checkpoint_path = os.path.join(logdir, 'checkpoints')
    path = os.path.join(checkpoint_path, f"{current_step}")
    abs_path = os.path.abspath(path)
    orbax_checkpointer.save(abs_path, params, force=True, save_args=save_args)
    


# Setup training function
randomizer = reachbot_randomize
train_fn = functools.partial(
    ppo.train, 
    **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    policy_params_fn=policy_params_fn,
    max_devices_per_host=1,
    log_training_metrics=True,
)

# Run training
print("Training the model...")
try:
    make_inference_fn, params, metrics = train_fn(
        environment=env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )
    print("Training completed successfully!")
except Exception as e:
    import traceback
    run_duration = str(datetime.now() - times[0])
    send_message_sync(
        task="Cave Exploration RL Training",
        duration=run_duration,
        result=f"Failed: {e}"
    )
    traceback.print_exc()
    raise

print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

# Save results
results_path = os.path.join(logdir, 'results.txt')
with open(results_path, 'w') as f:
    for i in range(len(total_rewards)):
        f.write(f"step: {timesteps[i]}, reward: {total_rewards[i]}, reward_std: {total_rewards_std[i]}\n")
    f.write(f"Time to jit: {times[1] - times[0]}\n")
    f.write(f"Time to train: {times[-1] - times[1]}\n")

# Save rewards as JSON
def nest_flat_dict(flat_dict):
    nested_dict = {}
    for key, value in flat_dict.items():
        parts = key.split('/')
        d = nested_dict
        for i, part in enumerate(parts):
            is_last_part = (i == len(parts) - 1)
            if is_last_part:
                if isinstance(d.get(part), dict):
                    d[part]['value'] = value
                else:
                    d[part] = value
            else:
                if not isinstance(d.get(part), dict):
                    d[part] = {'value': d[part]} if part in d else {}
                d = d[part]
    return nested_dict

nested_rewards = [nest_flat_dict(r) for r in rewards]
rewards_path = os.path.join(logdir, 'rewards.json')
with open(rewards_path, 'w') as fp:
    json.dump(nested_rewards, fp, indent=4, cls=JaxArrayEncoder)

# Save trained parameters
params_path = os.path.join(logdir, 'params')
model.save_params(params_path, params)

print(f"Training completed! Results saved to: {logdir}")
print(f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}")

# Store these variables for the video generation cell
trained_params = params
trained_make_inference_fn = make_inference_fn
trained_env = env
trained_logdir = logdir
# Video creation from trained model
# Free up training memory before rendering
del train_fn, network_factory, writer
gc.collect()

# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 5
rollout_steps = 5000


# Rollout policy and record simulation

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
episode_rewards = []

for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout = [state]  # Reset rollout for each episode
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps}, Current reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    episode_rewards.append(episode_reward)
    print(f"Episode {episode + 1} completed with {len(rollout)} states and total reward: {episode_reward:.3f}")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_ep._{episode}_reward_{episode_reward:.1f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully: Episode {episode + 1}, Reward: {episode_reward:.3f}")

# Print summary of all episodes
print("\n=== EPISODE REWARD SUMMARY ===")
for i, reward in enumerate(episode_rewards):
    print(f"Episode {i + 1}: {reward:.3f}")
print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.3f}")
print(f"Best episode: {episode_rewards.index(max(episode_rewards)) + 1} with reward {max(episode_rewards):.3f}")
print(f"Worst episode: {episode_rewards.index(min(episode_rewards)) + 1} with reward {min(episode_rewards):.3f}")


# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    result = f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    result = "No rewards recorded."

send_message_sync(
    task="Cave Exploration RL Training",
    duration=run_duration,
    result=result
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Video file: {video_path}")
print(f"Training duration: {run_duration}")
print(f"Final result: {result}")

# Output video render

In [ ]:
# Video creation from trained model


# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=False)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 5
rollout_steps = 5000


# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
episode_rewards = []

for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout = [state]  # Reset rollout for each episode
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps}, Current reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    episode_rewards.append(episode_reward)
    print(f"Episode {episode + 1} completed with {len(rollout)} states and total reward: {episode_reward:.3f}")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_episode_{episode}_reward_{episode_reward:.1f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully: Episode {episode + 1}, Reward: {episode_reward:.3f}")

# Print summary of all episodes
print("\n=== EPISODE REWARD SUMMARY ===")
for i, reward in enumerate(episode_rewards):
    print(f"Episode {i + 1}: {reward:.3f}")
print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.3f}")
print(f"Best episode: {episode_rewards.index(max(episode_rewards)) + 1} with reward {max(episode_rewards):.3f}")
print(f"Worst episode: {episode_rewards.index(min(episode_rewards)) + 1} with reward {min(episode_rewards):.3f}")

# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    training_result = f"Training final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    training_result = "No training rewards recorded."

# Include episode rewards in notification
episode_summary = f"Episode rewards: {[f'{r:.1f}' for r in episode_rewards]}, Avg: {sum(episode_rewards)/len(episode_rewards):.1f}"

send_message_sync(
    task="Cave Exploration RL Training",
    duration=run_duration,
    result=f"{training_result}\n{episode_summary}"
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Training duration: {run_duration}")
print(f"Training result: {training_result}")
print(f"Episode summary: {episode_summary}")